In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/0w/j8jyj1vs7fd1hh9rpnk57syw0000gn/T/ipykernel_16114/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

In [8]:
#In this lab, you will build a natural language processing model to classify SMS messages as either SPAM or HAM. Follow these steps:

#1. Load and explore the SMS dataset
#2. Preprocess the text data (cleaning, tokenization, removing stop words)
#3. Perform feature extraction (TF-IDF or Count Vectorization)
#4. Split the data into training and testing sets
#5. Train a classification model (Naive Bayes, Logistic Regression, etc.)
#6. Evaluate model performance using accuracy, precision, recall, and F1-score
#7. Test the model with new SMS messages

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [10]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("/Users/Shyam/Desktop/Ironhack-Bootcamp/Week 7/D1/lab-natural-language-processing/data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [12]:
from sklearn.model_selection import train_test_split

# Split data into features (X) and target (y)
X = data['text']   # Text messages
y = data['label']  # Labels (1 = spam, 0 = ham)

# Split into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Ensures balanced class distribution
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())

Training set size: 800
Testing set size: 200

Class distribution in training set:
label
0    446
1    354
Name: count, dtype: int64


## Data Preprocessing

In [17]:
import nltk

# Download only the required packages for spam detection
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /Users/Shyam/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /Users/Shyam/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/Shyam/nltk_data...


True

In [18]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [19]:
import re

def clean_html(text):
    """Remove HTML tags and clean the text"""
    
    # Remove inline JavaScript/CSS
    text = re.sub(r'<script.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?</style>', '', text, flags=re.DOTALL)
    
    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    return text

# Apply cleaning to your text data
X_train_clean = X_train.apply(clean_html)
X_test_clean = X_test.apply(clean_html)

# Check the result
print("Before cleaning:")
print(X_train.iloc[0][:200])
print("\nAfter cleaning:")
print(X_train_clean.iloc[0][:200])

Before cleaning:
Dear=2C Good day hope fine=2Cdear am writting this mail with due respect and heartful of tears since we have not known or met ourselves previously I am asking for your assistance=2Ci have will be very

After cleaning:
Dear=2C Good day hope fine=2Cdear am writting this mail with due respect and heartful of tears since we have not known or met ourselves previously I am asking for your assistance=2Ci have will be very


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [20]:
import re

def clean_text(text):
    """Clean text by removing special characters, numbers, etc."""
    
    # Remove prefixed 'b'
    text = re.sub(r'^b\s+', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove single characters
    text = re.sub(r'\s+[a-z]\s+', ' ', text)
    
    # Remove single characters from the start
    text = re.sub(r'^[a-z]\s+', '', text)
    
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading and trailing spaces
    text = text.strip()
    
    return text

# Apply cleaning to your data
X_train_clean = X_train_clean.apply(clean_text)
X_test_clean = X_test_clean.apply(clean_text)

# Check the result
print("Sample cleaned text:")
print(X_train_clean.iloc[0][:200])

Sample cleaned text:
dearc good day hope finecdear am writting this mail with due respect and heartful of tears since we have not known or met ourselves previously am asking for your assistanceci have will be very glad if


## Now let's work on removing stopwords
Remove the stopwords.

In [21]:
from nltk.corpus import stopwords

def remove_stopwords(text):
    """Remove stopwords from text"""
    
    # Get English stopwords
    stop_words = set(stopwords.words('english'))
    
    # Split text into words
    words = text.split()
    
    # Remove stopwords
    filtered_words = [word for word in words if word not in stop_words]
    
    # Join words back into text
    text = ' '.join(filtered_words)
    
    return text

# Apply stopword removal to your data
X_train_clean = X_train_clean.apply(remove_stopwords)
X_test_clean = X_test_clean.apply(remove_stopwords)

# Check the result
print("Sample text after removing stopwords:")
print(X_train_clean.iloc[0][:200])
print(f"\nNumber of words before: {len(X_train.iloc[0].split())}")
print(f"Number of words after: {len(X_train_clean.iloc[0].split())}")

Sample text after removing stopwords:
dearc good day hope finecdear writting mail due respect heartful tears since known met previously asking assistanceci glad render assistance situation nowe make proposal well known given opportunitye 

Number of words before: 287
Number of words after: 144


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [22]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def lemmatize_text(text):
    """Lemmatize text to reduce words to their base form"""
    
    # Initialize lemmatizer
    lemmatizer = WordNetLemmatizer()
    
    # Split text into words
    words = text.split()
    
    # Lemmatize each word
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    # Join words back into text
    text = ' '.join(lemmatized_words)
    
    return text

# Apply lemmatization to your data
X_train_clean = X_train_clean.apply(lemmatize_text)
X_test_clean = X_test_clean.apply(lemmatize_text)

# Check the result
print("Sample text after lemmatization:")
print(X_train_clean.iloc[0][:200])
print("\nExample transformations:")
print("running -> run")
print("better -> good") 
print("cats -> cat")

Sample text after lemmatization:
dearc good day hope finecdear writting mail due respect heartful tear since known met previously asking assistanceci glad render assistance situation nowe make proposal well known given opportunitye w

Example transformations:
running -> run
better -> good
cats -> cat


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Create CountVectorizer
vectorizer = CountVectorizer()

# Fit and transform the training data
X_train_bow = vectorizer.fit_transform(X_train_clean)

# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Separate spam and ham messages
spam_messages = X_train_clean[y_train == 1]
ham_messages = X_train_clean[y_train == 0]

# Get word counts for spam
spam_bow = vectorizer.transform(spam_messages)
spam_word_counts = spam_bow.sum(axis=0).A1
spam_word_freq = pd.DataFrame({'word': feature_names, 'count': spam_word_counts})
spam_word_freq = spam_word_freq.sort_values('count', ascending=False).head(10)

# Get word counts for ham
ham_bow = vectorizer.transform(ham_messages)
ham_word_counts = ham_bow.sum(axis=0).A1
ham_word_freq = pd.DataFrame({'word': feature_names, 'count': ham_word_counts})
ham_word_freq = ham_word_freq.sort_values('count', ascending=False).head(10)

# Display results
print("Top 10 words in SPAM messages:")
print(spam_word_freq)
print("\n" + "="*50 + "\n")
print("Top 10 words in HAM messages:")
print(ham_word_freq)

Top 10 words in SPAM messages:
              word  count
9735         money    708
140        account    596
1667          bank    571
6340          fund    540
2395      business    383
15550  transaction    327
3718       country    319
9571       million    305
3210       company    299
15613     transfer    293


Top 10 words in HAM messages:
            word  count
16864      would     91
11923  president     90
11444    percent     76
2488        call     75
14232      state     72
9877          mr     70
16819       work     70
10910        one     62
736     american     60
10623      obama     60


## Extra features

In [28]:
# Create data_train DataFrame with preprocessed text
data_train = pd.DataFrame({
    'preprocessed_text': X_train_clean,
    'label': y_train
})

# Create data_val DataFrame (validation/test set)
data_val = pd.DataFrame({
    'preprocessed_text': X_test_clean,
    'label': y_test
}).reset_index(drop=True)

In [29]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
442,dearc good day hope finecdear writting mail du...,1,1,1,1024
962,mr henry kaborethe chief auditor inchargeforei...,1,0,1,1954
971,,0,0,0,0
190,desk dradamu ismalerauditing accounting manage...,1,1,1,390
551,dear friend name loi cestradathe wife mr josep...,1,1,1,1507


## How would work the Bag of Words with Count Vectorizer concept?

In [32]:
## How would work the Bag of Words with Count Vectorizer concept?

#**Bag of Words (BoW)** is a text representation technique that converts text into numerical features by counting word occurrences.

#**How CountVectorizer Works:**

#1. **Create Vocabulary**: Scans all documents and creates a list of unique words
#2. **Count Occurrences**: For each document, counts how many times each word appears
#3. **Create Matrix**: Builds a matrix where each row is a document and each column is a word count

#**Example:**

from sklearn.feature_extraction.text import CountVectorizer

# Sample texts
texts = [
    "I love machine learning",
    "I love python programming",
    "machine learning is amazing"
]

# Create CountVectorizer
vectorizer = CountVectorizer()

# Fit and transform
bow_matrix = vectorizer.fit_transform(texts)

# View vocabulary
print("Vocabulary:", vectorizer.get_feature_names_out())
# Output: ['amazing' 'is' 'learning' 'love' 'machine' 'programming' 'python']

# View the matrix
print("\nBag of Words Matrix:")
print(bow_matrix.toarray())
# Output:
# [[0 0 1 1 1 0 0]  <- "I love machine learning"
#  [0 0 0 1 0 1 1]  <- "I love python programming"
#  [1 1 1 0 1 0 0]] <- "machine learning is amazing"

#**Key Points:**
#- Each number represents how many times that word appears in the document
#- Word order is lost (hence "bag" of words)
#- Converts text into numbers that machine learning models can understand

Vocabulary: ['amazing' 'is' 'learning' 'love' 'machine' 'programming' 'python']

Bag of Words Matrix:
[[0 0 1 1 1 0 0]
 [0 0 0 1 0 1 1]
 [1 1 1 0 1 0 0]]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Vectorize the training dataset
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)

# Vectorize the test dataset (use transform, not fit_transform)
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

# Print the shape of the vectorized datasets
print("Training set shape:", X_train_tfidf.shape)
print("Test set shape:", X_test_tfidf.shape)
print("\nNumber of features (unique words):", len(tfidf_vectorizer.get_feature_names_out()))

Training set shape: (800, 5000)
Test set shape: (200, 5000)

Number of features (unique words): 5000


## And the Train a Classifier?

In [35]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Initialize the classifier
classifier = MultinomialNB()

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Make predictions on test set
y_pred = classifier.predict(X_test_tfidf)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['HAM', 'SPAM']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.975

Classification Report:
              precision    recall  f1-score   support

         HAM       0.99      0.96      0.98       112
        SPAM       0.96      0.99      0.97        88

    accuracy                           0.97       200
   macro avg       0.97      0.98      0.97       200
weighted avg       0.98      0.97      0.98       200


Confusion Matrix:
[[108   4]
 [  1  87]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [37]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# Load test data (replace with actual Kaggle test file)
test_data = pd.read_csv("/Users/Shyam/Desktop/Ironhack-Bootcamp/Week 7/D1/lab-natural-language-processing/data/kg_test.csv", encoding='latin-1')
test_data.fillna("", inplace=True)

# Apply same preprocessing to test data
# (Use the same cleaning functions you created earlier)
X_test_kaggle = test_data['text'].apply(clean_html).apply(clean_text).apply(remove_stopwords).apply(lemmatize_text)

# OPTION 1: Bag of Words only
vectorizer_bow = CountVectorizer(max_features=5000)
X_train_bow = vectorizer_bow.fit_transform(X_train_clean)
X_test_bow = vectorizer_bow.transform(X_test_kaggle)

model1 = MultinomialNB()
model1.fit(X_train_bow, y_train)
predictions1 = model1.predict(X_test_bow)

# OPTION 2: TF-IDF only
vectorizer_tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train_clean)
X_test_tfidf = vectorizer_tfidf.transform(X_test_kaggle)

model2 = MultinomialNB()
model2.fit(X_train_tfidf, y_train)
predictions2 = model2.predict(X_test_tfidf)

# OPTION 3: TF-IDF + extra features
from scipy.sparse import hstack

# Create extra features for test data
test_features = pd.DataFrame({
    'preprocessed_text': X_test_kaggle
})
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

test_features['money_mark'] = test_features['preprocessed_text'].str.contains(money_simbol_list)*1
test_features['suspicious_words'] = test_features['preprocessed_text'].str.contains(suspicious_words)*1
test_features['text_len'] = test_features['preprocessed_text'].apply(lambda x: len(x))

# Combine TF-IDF with extra features
X_train_combined = hstack([X_train_tfidf, data_train[['money_mark', 'suspicious_words', 'text_len']].values])
X_test_combined = hstack([X_test_tfidf, test_features[['money_mark', 'suspicious_words', 'text_len']].values])

model3 = MultinomialNB()
model3.fit(X_train_combined, y_train)
predictions3 = model3.predict(X_test_combined)

# Save submissions
pd.DataFrame({'id': test_data.index, 'label': predictions1}).to_csv('submission_bow.csv', index=False)
pd.DataFrame({'id': test_data.index, 'label': predictions2}).to_csv('submission_tfidf.csv', index=False)
pd.DataFrame({'id': test_data.index, 'label': predictions3}).to_csv('submission_combined.csv', index=False)

print("✅ All submissions created! Test each one on Kaggle to see which performs best.")

✅ All submissions created! Test each one on Kaggle to see which performs best.
